In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import os

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

COLORS = {
    'Doc': '#4C72B0', 'Img': '#DD8452', 'Movie': '#55A868',
    'Rec': '#C44E52', 'BGM': '#8172B3'
}
DPI = 300

# MPLC weights data (from App/backend/services/mplc_weights.py)
# rerank 제거 (모든 도메인에서 0)
FEATURES = ["dense", "sparse", "asf", "keyword_count", "filename_substr", "z_dense"]

MPLC_WEIGHTS = {
    "doc": {
        "weights": {"dense": 14.396, "sparse": 0.0, "asf": 0.0,
                     "keyword_count": 3.096, "filename_substr": 0.276, "z_dense": 8.497},
        "cv_auc": 0.985
    },
    "image": {
        "weights": {"dense": 17.795, "sparse": 0.0, "asf": 0.0,
                     "keyword_count": 3.274, "filename_substr": 0.0, "z_dense": 0.315},
        "cv_auc": 0.921
    },
    "video": {
        "weights": {"dense": 9.753, "sparse": 2.675, "asf": 0.837,
                     "keyword_count": 0.124, "filename_substr": 0.0, "z_dense": 1.514},
        "cv_auc": 0.986
    },
    "audio": {
        "weights": {"dense": 0.539, "sparse": 0.234, "asf": 2.182,
                     "keyword_count": 3.561, "filename_substr": 0.0, "z_dense": 3.682},
        "cv_auc": 0.989
    },
    "bgm": {
        "weights": {"dense": 12.542, "sparse": 0.0, "asf": 0.0,
                     "keyword_count": 1.040, "filename_substr": 0.0, "z_dense": 0.0},
        "cv_auc": 0.922
    }
}

In [2]:
# fig05_mplc_weights_heatmap.png
# Annotated heatmap of MPLC weights

DOMAIN_LABELS = ["Doc", "Image", "Video", "Audio", "BGM"]
DOMAIN_KEYS = ["doc", "image", "video", "audio", "bgm"]

weight_matrix = np.array([
    [MPLC_WEIGHTS[dk]["weights"][f] for f in FEATURES]
    for dk in DOMAIN_KEYS
])

# Build masked colormap: zero values should appear white
# Use a copy of the colormap and set under color to white
from matplotlib.colors import Normalize
import matplotlib.cm as cm

cmap = plt.get_cmap('YlOrRd').copy()
cmap.set_under('white')

vmin = 1e-6  # just above zero so zeros fall under
vmax = weight_matrix.max()

fig, ax = plt.subplots(figsize=(12, 6))

im = ax.imshow(weight_matrix, cmap=cmap, aspect='auto', vmin=vmin, vmax=vmax)

ax.set_xticks(range(len(FEATURES)))
ax.set_xticklabels(FEATURES, fontsize=12)
ax.set_yticks(range(len(DOMAIN_LABELS)))
ax.set_yticklabels(DOMAIN_LABELS, fontsize=12)

# Annotate each cell
for i in range(len(DOMAIN_KEYS)):
    for j in range(len(FEATURES)):
        val = weight_matrix[i, j]
        if val == 0.0:
            text = "0"
            color = '#aaaaaa'
        elif val < 1.0:
            text = f"{val:.2f}"
            color = 'black'
        else:
            text = f"{val:.1f}"
            color = 'black' if val < vmax * 0.6 else 'white'
        ax.text(j, i, text, ha='center', va='center', fontsize=10, color=color, fontweight='bold')

plt.colorbar(im, ax=ax, label='가중치')
ax.set_title("MPLC 도메인별 Feature 가중치", fontsize=15, fontweight='bold', pad=14)
plt.tight_layout()

out_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'fig05_mplc_weights_heatmap.png')
plt.savefig(out_path, dpi=DPI, bbox_inches='tight')
plt.show()
print(f"Saved: {out_path}")

Saved: C:\yssong\KDT-FT-team3-Chainers\DB_insight\Figures\fig05_mplc_weights_heatmap.png


C:\Users\sjowu\AppData\Local\Temp\ipykernel_89268\1067681678.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# fig05_cv_auc_barplot.png
# Horizontal bar chart of CV AUC per domain

DOMAIN_DISPLAY = ["Doc", "Image", "Video", "Audio", "BGM"]
DOMAIN_KEYS_AUC = ["doc", "image", "video", "audio", "bgm"]
COLOR_MAP = {'doc': 'Doc', 'image': 'Img', 'video': 'Movie', 'audio': 'Rec', 'bgm': 'BGM'}

auc_values = [MPLC_WEIGHTS[dk]["cv_auc"] for dk in DOMAIN_KEYS_AUC]
bar_colors = [COLORS[COLOR_MAP[dk]] for dk in DOMAIN_KEYS_AUC]

fig, ax = plt.subplots(figsize=(10, 6))

y_pos = range(len(DOMAIN_DISPLAY))
bars = ax.barh(list(y_pos), auc_values, color=bar_colors, edgecolor='white', height=0.55)

# Vertical dashed baseline at 0.9
ax.axvline(x=0.9, color='gray', linestyle='--', linewidth=1.5, label='기준선 (0.9)')

# Annotate each bar with AUC value
for bar, val in zip(bars, auc_values):
    ax.text(
        val + 0.002, bar.get_y() + bar.get_height() / 2,
        f"{val:.3f}", va='center', ha='left', fontsize=12, fontweight='bold'
    )

ax.set_yticks(list(y_pos))
ax.set_yticklabels(DOMAIN_DISPLAY, fontsize=12)
ax.set_xlabel("CV AUC", fontsize=12)
ax.set_xlim(0.85, 1.005)
ax.set_title("MPLC Cross-Validation AUC (도메인별)", fontsize=15, fontweight='bold', pad=14)
ax.legend(fontsize=11)
ax.grid(axis='x', linestyle=':', alpha=0.5)
plt.tight_layout()

out_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'fig05_cv_auc_barplot.png')
plt.savefig(out_path, dpi=DPI, bbox_inches='tight')
plt.show()
print(f"Saved: {out_path}")

Saved: C:\yssong\KDT-FT-team3-Chainers\DB_insight\Figures\fig05_cv_auc_barplot.png


C:\Users\sjowu\AppData\Local\Temp\ipykernel_89268\982615420.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# fig05_feature_importance_radar.png
# Radar/spider chart — rerank 제거 (6 features), 데이터 포인트(점) 추가

DOMAIN_KEYS_RADAR = ["doc", "image", "video", "audio", "bgm"]
DOMAIN_DISPLAY_RADAR = ["Doc", "Image", "Video", "Audio", "BGM"]
COLOR_MAP_RADAR = {'doc': 'Doc', 'image': 'Img', 'video': 'Movie', 'audio': 'Rec', 'bgm': 'BGM'}
MARKERS = ['o', 's', 'D', '^', 'v']  # 도메인별 마커

# Build weight matrix (rerank 제외된 FEATURES 사용)
raw_matrix = np.array([
    [MPLC_WEIGHTS[dk]["weights"][f] for f in FEATURES]
    for dk in DOMAIN_KEYS_RADAR
])  # shape: (5, 6)

# Normalize per feature (column) to [0, 1]
col_max = raw_matrix.max(axis=0)
col_max[col_max == 0] = 1.0
norm_matrix = raw_matrix / col_max

# Radar setup
N = len(FEATURES)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

FEATURE_LABELS = ["dense", "sparse", "asf", "keyword\ncount", "filename\nsubstr", "z_dense"]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

for i, (dk, label) in enumerate(zip(DOMAIN_KEYS_RADAR, DOMAIN_DISPLAY_RADAR)):
    values = norm_matrix[i].tolist()
    values += values[:1]
    color = COLORS[COLOR_MAP_RADAR[dk]]
    ax.plot(angles, values, color=color, linewidth=2, linestyle='solid', label=label)
    ax.fill(angles, values, color=color, alpha=0.10)
    # 데이터 포인트(점) 추가
    ax.scatter(angles, values, color=color, s=50, marker=MARKERS[i],
               zorder=5, edgecolors='white', linewidths=0.8)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(FEATURE_LABELS, fontsize=11)

ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], fontsize=8, color='gray')
ax.set_ylim(0, 1.05)

ax.set_title("도메인별 Feature 중요도 레이더 차트", fontsize=15, fontweight='bold', pad=25)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.15), fontsize=12)

plt.tight_layout()

out_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'fig05_feature_importance_radar.png')
plt.savefig(out_path, dpi=DPI, bbox_inches='tight')
plt.show()
print(f"Saved: {out_path}")